# Build the facility heat risk dataset and insert it into DB

This notebook builds a simple data pipeline for SILO climate data and writes a cleaned CSV.

## Data source
- SILO Data Drill API (Queensland Government, CC BY 4.0)
- https://www.longpaddock.qld.gov.au/silo/api-documentation/reference/
- Daily max + min temperature, 3 years (2022–2024), per facility grid cell (~5 km)

## Pipeline rules
- read facility coordinates from `aged_care_services` in the DB
- skip facilities with missing latitude/longitude
- round coordinates to SILO's 0.05° grid before querying
- cache responses per grid cell so duplicate cells aren't fetched twice
- fetch in parallel with a small worker pool to be polite to the public API
- compute four heat metrics per facility:
  - `avg_hot_days_per_year` — days with max temp ≥ 35 °C
  - `avg_extreme_days_per_year` — days with max temp ≥ 40 °C
  - `avg_heatwaves_per_year` — runs of 3+ consecutive days ≥ 35 °C
  - `avg_hot_nights_per_year` — nights with min temp ≥ 20 °C
- combine the four metrics into a composite `risk_score` (0–2)
- bucket into a `heat_risk` label (`low`, `medium`, `high`)
- record failed facilities with `heat_risk = "error"` instead of dropping them
- write the result to a CSV


In [1]:
from pathlib import Path
import io
import os
import time
import requests
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

load_dotenv(".env.db", override=True)

# --- SILO settings ---
SILO_EMAIL = os.getenv("SILO_EMAIL", "test@gmail.com")
SILO_URL = "https://www.longpaddock.qld.gov.au/cgi-bin/silo/DataDrillDataset.php"

START_DATE = "20220101"
FINISH_DATE = "20241231"

# --- Output ---
OUTPUT_CSV = Path("input/facility_heat_risk.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- Parallelism ---
MAX_WORKERS = 8   # concurrent requests to SILO

OUTPUT_CSV


PosixPath('input/facility_heat_risk.csv')

In [2]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    facilities = pd.read_sql_query(
        "SELECT id, service_name, latitude, longitude FROM aged_care_services",
        conn,
    )

facilities.head()


,id,service_name,latitude,longitude
0,f52839f6-aad1-4530-a1b9-be65b55db10a,Braidwood Multi-Purpose Service,-35.442764,149.805407
1,b5807c0e-cdff-4898-8f01-3188de404e89,Dellacourt,-36.075401,146.890896
2,6682630d-b46b-417a-88a8-9a92cc3df8a6,Brerrina Multi-Purpose Service,-29.961591,146.864313
3,816ec8b3-dd89-47b6-ab28-38c84219067a,BaptistCare Maranoa Centre - Alstonville,-28.840536,153.436634
4,4fe8361b-e220-4ea6-8607-96e3e2c1ca23,Urana Multi-Purpose Service,-35.325391,146.267697


In [3]:
def fetch_silo(lat, lon):
    """Get daily max + min temperature for one location from SILO Data Drill."""
    params = {
        "lat": round(lat * 20) / 20,    # SILO grid is 0.05°
        "lon": round(lon * 20) / 20,
        "start": START_DATE,
        "finish": FINISH_DATE,
        "format": "csv",
        "comment": "XN",                 # X=max temp, N=min temp
        "username": SILO_EMAIL,
        "password": "apirequest",
    }
    r = requests.get(SILO_URL, params=params, timeout=60)
    r.raise_for_status()

    # SILO prepends metadata lines before the CSV header.
    # Find the real header row (the one containing "date" or "YYYY-MM-DD").
    lines = r.text.splitlines()
    header_idx = next(
        (i for i, line in enumerate(lines)
         if "date" in line.lower() or "yyyy-mm-dd" in line.lower()),
        None,
    )
    if header_idx is None:
        raise RuntimeError(f"Couldn't find header row. Response starts with:\n{r.text[:500]}")

    csv_text = "\n".join(lines[header_idx:])
    df = pd.read_csv(io.StringIO(csv_text))
    df.columns = [c.strip().lower() for c in df.columns]

    date_col = [c for c in df.columns if "date" in c or "yyyy" in c][0]
    max_col  = [c for c in df.columns if "max" in c][0]
    min_col  = [c for c in df.columns if "min" in c][0]

    return pd.DataFrame({
        "date":       pd.to_datetime(df[date_col]),
        "max_temp_c": pd.to_numeric(df[max_col], errors="coerce"),
        "min_temp_c": pd.to_numeric(df[min_col], errors="coerce"),
    }).dropna()


# Quick smoke test
test = fetch_silo(-37.80, 144.95)   # Melbourne
test.head()


,date,max_temp_c,min_temp_c
0,2022-01-01,37.8,19.7
1,2022-01-02,31.4,20.6
2,2022-01-03,21.5,16.2
3,2022-01-04,23.1,15.8
4,2022-01-05,29.6,17.8


In [4]:
def compute_metrics(df):
    """Return the four heat metrics from a daily temp dataframe."""
    n_years = df["date"].dt.year.nunique()

    # Count heatwaves (3+ consecutive days >= 35°C)
    hot = (df["max_temp_c"] >= 35).astype(int).values
    heatwaves, run = 0, 0
    for v in hot:
        if v:
            run += 1
        else:
            if run >= 3:
                heatwaves += 1
            run = 0
    if run >= 3:
        heatwaves += 1

    return {
        "avg_hot_days_per_year":     round((df["max_temp_c"] >= 35).sum() / n_years, 2),
        "avg_extreme_days_per_year": round((df["max_temp_c"] >= 40).sum() / n_years, 2),
        "avg_heatwaves_per_year":    round(heatwaves / n_years, 2),
        "avg_hot_nights_per_year":   round((df["min_temp_c"] >= 20).sum() / n_years, 2),
    }


# Test
compute_metrics(test)


{'avg_hot_days_per_year': np.float64(7.33),
 'avg_extreme_days_per_year': np.float64(0.33),
 'avg_heatwaves_per_year': 0.33,
 'avg_hot_nights_per_year': np.float64(4.67)}

In [5]:
def risk_label(m):
    """Composite score → low / medium / high.

    Each metric is normalised against a 'hot inland Australian climate' reference
    (~Mildura level), capped at 2x, then averaged.
    """
    score = (
        min(m["avg_hot_days_per_year"]     / 60, 2) +
        min(m["avg_extreme_days_per_year"] / 15, 2) +
        min(m["avg_heatwaves_per_year"]    /  8, 2) +
        min(m["avg_hot_nights_per_year"]   / 40, 2)
    ) / 4

    if score <= 0.35:
        return "low", round(score, 3)
    elif score <= 0.75:
        return "medium", round(score, 3)
    else:
        return "high", round(score, 3)


In [6]:
# Drop rows without coordinates
valid = facilities.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
print(f"Scoring {len(valid)} facilities with valid coordinates "
      f"(skipped {len(facilities) - len(valid)} without coords).")

# Cache by grid cell — many facilities share the same SILO cell (~5 km)
cell_cache: dict[tuple, dict] = {}
cache_lock = Lock()


def score_one(row):
    """Process one facility. Returns a result dict."""
    lat, lon = row["latitude"], row["longitude"]
    cell = (round(lat * 20) / 20, round(lon * 20) / 20)

    base = {
        "service_id": row["id"],
        "service_name": row.get("service_name"),
        "latitude": lat,
        "longitude": lon,
        "grid_lat": cell[0],
        "grid_lon": cell[1],
    }

    try:
        with cache_lock:
            cached = cell_cache.get(cell)

        if cached is not None:
            metrics = cached
        else:
            daily = fetch_silo(lat, lon)
            metrics = compute_metrics(daily)
            with cache_lock:
                cell_cache[cell] = metrics

        label, score = risk_label(metrics)
        return {**base, **metrics, "heat_risk": label, "risk_score": score, "error_message": None}

    except Exception as e:
        return {
            **base,
            "avg_hot_days_per_year": None,
            "avg_extreme_days_per_year": None,
            "avg_heatwaves_per_year": None,
            "avg_hot_nights_per_year": None,
            "heat_risk": "error",
            "risk_score": None,
            "error_message": str(e),
        }


results = []
done = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(score_one, row): i for i, row in valid.iterrows()}
    for future in as_completed(futures):
        results.append(future.result())
        done += 1
        if done % 25 == 0 or done == len(valid):
            print(f"  {done}/{len(valid)} done — cache size: {len(cell_cache)}")

results_df = pd.DataFrame(results)
print(f"\nFinished. {len(cell_cache)} unique SILO grid cells queried.")
results_df.head()


Scoring 3015 facilities with valid coordinates (skipped 0 without coords).
  25/3015 done — cache size: 24
  50/3015 done — cache size: 46
  75/3015 done — cache size: 65
  100/3015 done — cache size: 83
  125/3015 done — cache size: 105
  150/3015 done — cache size: 125
  175/3015 done — cache size: 142
  200/3015 done — cache size: 162
  225/3015 done — cache size: 182
  250/3015 done — cache size: 195
  275/3015 done — cache size: 207
  300/3015 done — cache size: 220
  325/3015 done — cache size: 229
  350/3015 done — cache size: 235
  375/3015 done — cache size: 237
  400/3015 done — cache size: 239
  425/3015 done — cache size: 242
  450/3015 done — cache size: 247
  475/3015 done — cache size: 249
  500/3015 done — cache size: 261
  525/3015 done — cache size: 272
  550/3015 done — cache size: 284
  575/3015 done — cache size: 287
  600/3015 done — cache size: 298
  625/3015 done — cache size: 313
  650/3015 done — cache size: 335
  675/3015 done — cache size: 352
  700/3015 don

,service_id,service_name,latitude,longitude,grid_lat,grid_lon,avg_hot_days_per_year,avg_extreme_days_per_year,avg_heatwaves_per_year,avg_hot_nights_per_year,heat_risk,risk_score,error_message
0,aa4d0a87-b70c-43d6-83bc-8550dba3a686,Macquarie Lodge Aged Care Plus Centre,-33.939656,151.134856,-33.95,151.15,4.33,0.33,0.00,43.00,low,0.292,NaN
1,9cc01186-064b-4b8c-a900-7e3cf97b3076,Uniting Autumn Lodge Butler Street,-30.503834,151.658370,-30.50,151.65,0.00,0.00,0.00,0.33,low,0.002,NaN
2,6682630d-b46b-417a-88a8-9a92cc3df8a6,Brerrina Multi-Purpose Service,-29.961591,146.864313,-29.95,146.85,79.00,11.00,10.00,68.00,high,1.250,NaN
3,4fe8361b-e220-4ea6-8607-96e3e2c1ca23,Urana Multi-Purpose Service,-35.325391,146.267697,-35.35,146.25,28.00,2.00,3.33,19.00,medium,0.373,NaN
4,816ec8b3-dd89-47b6-ab28-38c84219067a,BaptistCare Maranoa Centre - Alstonville,-28.840536,153.436634,-28.85,153.45,0.33,0.00,0.00,46.00,low,0.289,NaN


In [7]:
print(f"Total facilities scored: {len(results_df)}")
print(f"Errors: {int((results_df['heat_risk'] == 'error').sum())}")
print(f"Missing risk_score: {int(results_df['risk_score'].isna().sum())}")
print("\nHeat risk distribution:")
print(results_df["heat_risk"].value_counts())


Total facilities scored: 3015
Errors: 1
Missing risk_score: 1

Heat risk distribution:
heat_risk
low       2061
medium     817
high       136
error        1
Name: count, dtype: int64


In [8]:
# Attach the data window so the DB has provenance for each row
results_df["data_start_date"] = pd.to_datetime(START_DATE, format="%Y%m%d").date()
results_df["data_end_date"] = pd.to_datetime(FINISH_DATE, format="%Y%m%d").date()

results_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(results_df)} rows to {OUTPUT_CSV}")


Saved 3015 rows to input/facility_heat_risk.csv


# Insert Clean Data into DB

In [11]:
# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/facility_heat_risk.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)


Schema created or already exists.
carelink


In [12]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
# Read back from CSV so the insert step is fully decoupled from the fetch step.
df_insert = pd.read_csv(OUTPUT_CSV)

target_columns = [
    "service_id",
    "latitude",
    "longitude",
    "grid_lat",
    "grid_lon",
    "avg_hot_days_per_year",
    "avg_extreme_days_per_year",
    "avg_heatwaves_per_year",
    "avg_hot_nights_per_year",
    "risk_score",
    "heat_risk",    
    "data_start_date",
    "data_end_date",
]

df_insert = df_insert[target_columns].copy()

# text columns
text_cols = ["service_id", "heat_risk"]
for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# float columns
float_cols = [
    "latitude",
    "longitude",
    "grid_lat",
    "grid_lon",
    "avg_hot_days_per_year",
    "avg_extreme_days_per_year",
    "avg_heatwaves_per_year",
    "avg_hot_nights_per_year",
    "risk_score",
]
for col in float_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce")

# date columns
date_cols = ["data_start_date", "data_end_date"]
for col in date_cols:
    df_insert[col] = pd.to_datetime(df_insert[col], errors="coerce").dt.date

df_insert.head()


,service_id,latitude,longitude,grid_lat,grid_lon,avg_hot_days_per_year,avg_extreme_days_per_year,avg_heatwaves_per_year,avg_hot_nights_per_year,risk_score,heat_risk,data_start_date,data_end_date
0,aa4d0a87-b70c-43d6-83bc-8550dba3a686,-33.939656,151.134856,-33.95,151.15,4.33,0.33,0.00,43.00,0.292,low,2022-01-01,2024-12-31
1,9cc01186-064b-4b8c-a900-7e3cf97b3076,-30.503834,151.658370,-30.50,151.65,0.00,0.00,0.00,0.33,0.002,low,2022-01-01,2024-12-31
2,6682630d-b46b-417a-88a8-9a92cc3df8a6,-29.961591,146.864313,-29.95,146.85,79.00,11.00,10.00,68.00,1.250,high,2022-01-01,2024-12-31
3,4fe8361b-e220-4ea6-8607-96e3e2c1ca23,-35.325391,146.267697,-35.35,146.25,28.00,2.00,3.33,19.00,0.373,medium,2022-01-01,2024-12-31
4,816ec8b3-dd89-47b6-ab28-38c84219067a,-28.840536,153.436634,-28.85,153.45,0.33,0.00,0.00,46.00,0.289,low,2022-01-01,2024-12-31


In [13]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "facility_heat_risk",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into facility_heat_risk")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM facility_heat_risk"))
    row_count = result.scalar()

print("Rows in table:", row_count)


Inserted 3015 rows into facility_heat_risk
Rows in table: 3015
